In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import collections, csv, datetime, glob, hashlib, json, math, os, random, re, shutil
import signal, subprocess, sys, tempfile, time, traceback, uuid
from importlib import metadata

def digest(data):
    if isinstance(data, str):
        data = data.encode("utf-8")
    return hashlib.sha256(data).hexdigest()

def write_text(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_file() and path.read_text() == value:
        return
    temp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temp.write_text(value)
    os.replace(temp, path)

def write_json(path, value):
    write_text(path, json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n")

def command(argv, cwd=None, timeout=90, log=None):
    start = time.monotonic()
    status = "DONE"
    rc = None
    try:
        with subprocess.Popen(
            [str(arg) for arg in argv], cwd=cwd, stdin=subprocess.DEVNULL,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True,
        ) as process:
            try:
                raw, _ = process.communicate(timeout=timeout)
                rc = process.returncode
            except subprocess.TimeoutExpired:
                status = "TIMEOUT"
                os.killpg(process.pid, signal.SIGKILL)
                raw, _ = process.communicate()
            except BaseException:
                os.killpg(process.pid, signal.SIGKILL)
                process.communicate()
                raise
        detail = raw.decode("utf-8", errors="replace")
    except OSError as exc:
        status, detail = "TOOL_ERROR", str(exc)
    if log:
        Path(log).write_text(detail)
    return {"rc": rc, "status": status, "text": detail,
            "seconds": round(time.monotonic() - start, 4)}

def checked(argv, **kwargs):
    result = command(argv, **kwargs)
    if result["status"] != "DONE" or result["rc"] != 0:
        raise RuntimeError(f"{argv[0]}: {result['status']}\n{result['text'][-5000:]}")
    return result["text"]

def download(url, target):
    tmp = target.with_suffix(target.suffix + ".partial")
    checked(["curl", "-fLsS", "--retry", "2", "--max-time", "600", "-o", tmp, url], timeout=630)
    os.replace(tmp, target)

def install_python_dependencies(model_key):
    if not IN_COLAB:
        return
    from packaging.requirements import Requirement
    dependencies = ["pandas>=2.0", "matplotlib>=3.7"]
    if model_key is not None:
        dependencies += [
            "transformers>=4.51,<5", "accelerate>=1.6,<2", "bitsandbytes>=0.45.5,<1",
            "huggingface_hub>=0.30,<1", "safetensors>=0.4.3,<1", "jinja2>=3.1,<4",
        ]
    missing = []
    for spec in dependencies:
        requirement = Requirement(spec)
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            installed = None
        if installed is None or installed not in requirement.specifier:
            missing.append(spec)
    if missing:
        print("Installing Python dependencies:", ", ".join(missing), flush=True)
        checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *missing],
                timeout=600)
    if model_key is not None:
        import torch
        if not torch.cuda.is_available():
            print("CPU runtime. Select a GPU if the experiment has model inference left to run.")
            return
        from transformers import BitsAndBytesConfig
        import bitsandbytes
        BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4")
        print("GPU:", torch.cuda.get_device_name(0), "| bitsandbytes:", bitsandbytes.__version__)

def install_opensta_fallback():
    """Extract the Debian Bookworm amd64 OpenSTA build into the runtime."""
    if os.uname().machine != "x86_64":
        raise RuntimeError("The OpenSTA fallback requires an x86-64 runtime.")
    install_dir = RUNTIME / "opensta-bookworm"
    sta = install_dir / "usr" / "bin" / "sta"
    archive = RUNTIME / "opensta_0~20191111gitc018cb2+dfsg-1_amd64.deb"
    expected = "a7e48ea4fb0c3d96c3cf9f65514f6d5e557ebbd6f4a4b9017c6b511b117c7dbc"
    if not sta.is_file():
        if not archive.is_file():
            download("https://deb.debian.org/debian/pool/main/o/opensta/" + archive.name, archive)
        if digest(archive.read_bytes()) != expected:
            archive.unlink()
            raise RuntimeError("OpenSTA package checksum mismatch. Rerun setup to download it again.")
        checked(["dpkg-deb", "--extract", archive, install_dir], timeout=60)
    probe = command([sta, "-version"], timeout=30)
    if probe["rc"] != 0 and "libtcl8.6.so" in probe["text"]:
        checked(["apt-get", "install", "-y", "-qq", "libtcl8.6"], timeout=120)
    checked([sta, "-version"], timeout=30)
    os.environ["PATH"] = f"{sta.parent}:{os.environ['PATH']}"

def configure_tools():
    RUNTIME.mkdir(parents=True, exist_ok=True)
    tools = ("yosys", "yosys-abc", "iverilog", "vvp", "eqy", "sby")
    if IN_COLAB and INSTALL_TOOLS:
        suite = RUNTIME / "oss-cad-suite"
        if (suite / "bin" / "eqy").is_file():
            os.environ["PATH"] = f"{suite / 'bin'}:{os.environ['PATH']}"
        packages = [package for package in ("curl", "tar", "make") if not shutil.which(package)]
        need_sta = not (shutil.which("sta") or shutil.which("openroad"))
        if packages or need_sta:
            checked(["apt-get", "update", "-qq"], timeout=180)
        if packages:
            checked(["apt-get", "install", "-y", "-qq", *packages], timeout=300)
        if any(not shutil.which(tool) for tool in tools):
            version = "2026-11-17"
            archive = RUNTIME / f"oss-cad-suite-linux-x64-{version.replace('-', '')}.tgz"
            url = f"https://github.com/YosysHQ/oss-cad-suite-build/releases/download/{version}/{archive.name}"
            if not archive.exists():
                download(url, archive)
            checked(["tar", "-xzf", archive, "-C", RUNTIME], timeout=240)
            archive.unlink()
            os.environ["PATH"] = f"{suite / 'bin'}:{os.environ['PATH']}"
        if need_sta:
            try:
                checked(["apt-get", "install", "-y", "-qq", "opensta"], timeout=300)
            except RuntimeError:
                debs = sorted(PROJECT.glob("openroad*.deb"))
                if debs:
                    deb = RUNTIME / debs[0].name
                    shutil.copyfile(debs[0], deb)
                    checked(["apt-get", "install", "-y", "-qq", deb], timeout=300)
                else:
                    print("OpenSTA is unavailable from apt. Extracting the Debian package fallback.")
                    install_opensta_fallback()
    missing = [name for name in tools if not shutil.which(name)]
    if missing:
        raise RuntimeError("Required programs: " + ", ".join(missing))
    sta = shutil.which("sta")
    openroad = shutil.which("openroad")
    if not (sta or openroad):
        raise RuntimeError("Install OpenSTA (sta), or place your OpenROAD .deb in the project folder.")
    return sta, openroad

def configure_technology(openroad_only):
    platform = "https://raw.githubusercontent.com/The-OpenROAD-Project/OpenROAD-flow-scripts/master/flow/platforms/nangate45/"
    technology = RUNTIME / "nangate45"
    technology.mkdir(exist_ok=True)
    items = {"liberty": "lib/NangateOpenCellLibrary_typical.lib"}
    if openroad_only:
        items.update(tech_lef="lef/NangateOpenCellLibrary.tech.lef",
                     cells_lef="lef/NangateOpenCellLibrary.macro.mod.lef")
    files = {}
    for key, relative in items.items():
        filename = Path(relative).name
        local = technology / filename
        drive_source = PROJECT / "scripts" / filename
        drive_source.parent.mkdir(parents=True, exist_ok=True)
        if not drive_source.is_file():
            download(platform + relative, drive_source)
        if not local.is_file() or digest(local.read_bytes()) != digest(drive_source.read_bytes()):
            shutil.copyfile(drive_source, local)
        files[key] = local
    return files

def software_versions():
    commands = {"yosys": ["yosys", "-V"], "eqy": ["eqy", "--version"],
                "iverilog": ["iverilog", "-V"],
                "abc": [shutil.which("yosys-abc") or "yosys-abc", "-c", "version"],
                "timing": [STA_EXEC, "-version"] if STA_EXEC else [OPENROAD_EXEC, "-version"]}
    result = {}
    for name, argv in commands.items():
        detail = command(argv, timeout=20)["text"]
        result[name] = next((line.strip() for line in detail.splitlines() if line.strip()), "unknown")
    return result


In [ ]:
WORKFLOW = "rl"

try:
    import google.colab
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT = Path(os.environ.get("RTL_PROJECT", str(DRIVE_ROOT / f"rtl_opt_{WORKFLOW}_project")))
OTHER_WORKFLOW = "rl" if WORKFLOW == "llm" else "llm"
OTHER_PROJECT = Path(os.environ.get("RTL_OTHER_PROJECT", str(DRIVE_ROOT / f"rtl_opt_{OTHER_WORKFLOW}_project")))
LEGACY_PROJECT = DRIVE_ROOT / "rtl_opt_project"
RUN_TAG = "default"  # Change only when you want a separate experiment with identical settings.
PROJECT.mkdir(parents=True, exist_ok=True)
RUNTIME = Path(os.environ.get("RTL_RUNTIME", "/content/rtl_verified_runtime"))
INSTALL_TOOLS = True
MODEL_KEY = "coder_7b" if WORKFLOW == "llm" else None
# Options: coder_7b, qwen3_8b, coder_14b, codev_r1. None keeps RL on CPU.
CLOCK_NS = 20.0
INPUT_SLEW_NS = 0.05
OUTPUT_LOAD_PF = 0.005
MAX_AREA_FACTOR = 1.05
MAX_DELAY_FACTOR = 1.05
SIM_TIMEOUT_S = 60
FORMAL_TIMEOUT_S = 180
FORMAL_WORKERS = 2
SEED_BASE = 20261119
OUTPUT = PROJECT / "results" / "verified_rtl_v4" / WORKFLOW


In [ ]:
install_python_dependencies(MODEL_KEY)


In [ ]:
# Exact RTL snapshot from rtl_opt_project. MIT notice for RTLLM-derived sources is below.
DRIVE_SNAPSHOT = {
    'alu': r"""`timescale 1ns / 1ps

module alu(
    input [31:0] a,
    input [31:0] b,
    input [5:0] aluc,
    output [31:0] r,
    output zero,
    output carry,
    output negative,
    output overflow,
    output flag
    );
    

    parameter ADD = 6'b100000;
    parameter ADDU = 6'b100001;
    parameter SUB = 6'b100010;
    parameter SUBU = 6'b100011;
    parameter AND = 6'b100100;
    parameter OR = 6'b100101;
    parameter XOR = 6'b100110;
    parameter NOR = 6'b100111;
    parameter SLT = 6'b101010;
    parameter SLTU = 6'b101011;
    parameter SLL = 6'b000000;
    parameter SRL = 6'b000010;
    parameter SRA = 6'b000011;
    parameter SLLV = 6'b000100;
    parameter SRLV = 6'b000110;
    parameter SRAV = 6'b000111;
    parameter JR = 6'b001000;
    
    parameter LUI = 6'b001111;
      
    
    wire signed [31:0] a_signed;
    wire signed [31:0] b_signed;
    
  
    reg [32:0] res;
    
    
    assign a_signed = a;
    assign b_signed = b;
    assign r = res[31:0];
    
    assign flag = (aluc == SLT || aluc == SLTU) ? ((aluc == SLT) ? (a_signed < b_signed) : (a < b)) : 1'bz; 
    assign zero = (res == 32'b0) ? 1'b1 : 1'b0;
    
    always @ (a or b or aluc)
    begin
        case(aluc)
            ADD: begin
                res <= a_signed + b_signed;
            end
            ADDU: begin
                res <= a + b;
            end
            SUB: begin 
                res <= a_signed - b_signed;
            end
            SUBU: begin 
                res <= a - b;
            end
            AND: begin
                res <= a & b;
            end
            OR: begin
                res <= a | b;
            end
            XOR: begin
                res <= a ^ b;
            end
            NOR: begin
                res <= ~(a | b);
            end
            SLT: begin
                res <= a_signed < b_signed ? 1 : 0;
            end
            SLTU: begin
                res <= a < b ? 1 : 0;
            end
            SLL: begin
                res <= b << a;
            end
            SRL: begin
                res <= b >> a;
            end
            SRA: begin
                res <= b_signed >>> a_signed;
            end
            SLLV: begin
                res <= b << a[4:0];
            end
            SRLV: begin
                res <= b >> a[4:0];
            end
            SRAV: begin
                res <= b_signed >>> a_signed[4:0];
            end
            LUI: begin
                res <= {a[15:0], 16'h0000};
            end
            default:
            begin
                res <= 32'bz;
            end
        endcase
    end
endmodule
""",
    'multiplier': r"""module multi_8bit (
  input [7:0] A,
  input [7:0] B,
  output reg [15:0] product
);

  reg [7:0] multiplicand;
  reg [3:0] shift_count;

  always @* begin
    product = 16'b0; 
    multiplicand = A; 
    shift_count = 0;  

    for (int i = 0; i < 8; i = i + 1) begin
      if (B[i] == 1) begin
        product = product + (multiplicand << shift_count);
      end
      shift_count = shift_count + 1;
    end
  end

endmodule""",
    'shifter': r"""module barrel_shifter (in, ctrl, out);
  input  [7:0] in;
  input [2:0] ctrl;
  output [7:0] out;
  wire [7:0] x,y;
 
//4bit shift right
mux2X1  ins_17 (.in0(in[7]),.in1(1'b0),.sel(ctrl[2]),.out(x[7]));
mux2X1  ins_16 (.in0(in[6]),.in1(1'b0),.sel(ctrl[2]),.out(x[6]));
mux2X1  ins_15 (.in0(in[5]),.in1(1'b0),.sel(ctrl[2]),.out(x[5]));
mux2X1  ins_14 (.in0(in[4]),.in1(1'b0),.sel(ctrl[2]),.out(x[4]));
mux2X1  ins_13 (.in0(in[3]),.in1(in[7]),.sel(ctrl[2]),.out(x[3]));
mux2X1  ins_12 (.in0(in[2]),.in1(in[6]),.sel(ctrl[2]),.out(x[2]));
mux2X1  ins_11 (.in0(in[1]),.in1(in[5]),.sel(ctrl[2]),.out(x[1]));
mux2X1  ins_10 (.in0(in[0]),.in1(in[4]),.sel(ctrl[2]),.out(x[0]));
 
//2 bit shift right
 
mux2X1  ins_27 (.in0(x[7]),.in1(1'b0),.sel(ctrl[1]),.out(y[7]));
mux2X1  ins_26 (.in0(x[6]),.in1(1'b0),.sel(ctrl[1]),.out(y[6]));
mux2X1  ins_25 (.in0(x[5]),.in1(x[7]),.sel(ctrl[1]),.out(y[5]));
mux2X1  ins_24 (.in0(x[4]),.in1(x[6]),.sel(ctrl[1]),.out(y[4]));
mux2X1  ins_23 (.in0(x[3]),.in1(x[5]),.sel(ctrl[1]),.out(y[3]));
mux2X1  ins_22 (.in0(x[2]),.in1(x[4]),.sel(ctrl[1]),.out(y[2]));
mux2X1  ins_21 (.in0(x[1]),.in1(x[3]),.sel(ctrl[1]),.out(y[1]));
mux2X1  ins_20 (.in0(x[0]),.in1(x[2]),.sel(ctrl[1]),.out(y[0]));
 
//1 bit shift right
mux2X1  ins_07 (.in0(y[7]),.in1(1'b0),.sel(ctrl[0]),.out(out[7]));
mux2X1  ins_06 (.in0(y[6]),.in1(y[7]),.sel(ctrl[0]),.out(out[6]));
mux2X1  ins_05 (.in0(y[5]),.in1(y[6]),.sel(ctrl[0]),.out(out[5]));
mux2X1  ins_04 (.in0(y[4]),.in1(y[5]),.sel(ctrl[0]),.out(out[4]));
mux2X1  ins_03 (.in0(y[3]),.in1(y[4]),.sel(ctrl[0]),.out(out[3]));
mux2X1  ins_02 (.in0(y[2]),.in1(y[3]),.sel(ctrl[0]),.out(out[2]));
mux2X1  ins_01 (.in0(y[1]),.in1(y[2]),.sel(ctrl[0]),.out(out[1]));
mux2X1  ins_00 (.in0(y[0]),.in1(y[1]),.sel(ctrl[0]),.out(out[0]));

endmodule

module mux2X1( in0,in1,sel,out);
input in0,in1;
input sel;
output out;
assign out=(sel)?in1:in0;
endmodule""",
    'counter': r"""`timescale 1ns/1ps
module counter_12 
(
  input rst_n,
  input clk,
  input valid_count,

  output reg [3:0] out
);

  always @(posedge clk or negedge rst_n) begin
    if (!rst_n) 
	begin
      out <= 4'b0000;
    end 

	else if (valid_count) 
	begin
      if (out == 4'd11) 
	  begin
        out <= 4'b0000;
      end 
	  else begin
        out <= out + 1;
      end
    end 
	
	else begin
      out <= out; // Pause the count when valid_count is invalid
    end
  end

endmodule
""",
    'arbiter': r"""module arbiter(input [7:0] req, output reg [7:0] grant);
  integer i;
  always @* begin
    grant = 8'd0;
    for (i = 7; i >= 0; i = i - 1)
      if (req[i]) grant = (8'b1 << i);
  end
endmodule
""",
    'fifo': r"""module fifo(input clk, input rst, input wr_en, input rd_en,
  input [7:0] din, output reg [7:0] dout, output full, output empty);
  reg [7:0] mem [0:3];
  reg [1:0] wptr, rptr;
  reg [2:0] count;
  assign full = (count == 3'd4);
  assign empty = (count == 3'd0);
  always @(posedge clk) begin
    if (rst) begin
      wptr <= 0; rptr <= 0; count <= 0; dout <= 0;
    end else begin
      if (wr_en && !full) begin
        mem[wptr] <= din;
        wptr <= wptr + 1'b1;
      end
      if (rd_en && !empty) begin
        dout <= mem[rptr];
        rptr <= rptr + 1'b1;
      end
      case ({(wr_en && !full), (rd_en && !empty)})
        2'b10: count <= count + 1'b1;
        2'b01: count <= count - 1'b1;
        default: count <= count;
      endcase
    end
  end
endmodule
""",
}
ALU_UPSTREAM = r"""`timescale 1ns / 1ps

module verified_alu(
    input [31:0] a,
    input [31:0] b,
    input [5:0] aluc,
    output [31:0] r,
    output zero,
    output carry,
    output negative,
    output overflow,
    output flag
    );
    

    parameter ADD = 6'b100000;
    parameter ADDU = 6'b100001;
    parameter SUB = 6'b100010;
    parameter SUBU = 6'b100011;
    parameter AND = 6'b100100;
    parameter OR = 6'b100101;
    parameter XOR = 6'b100110;
    parameter NOR = 6'b100111;
    parameter SLT = 6'b101010;
    parameter SLTU = 6'b101011;
    parameter SLL = 6'b000000;
    parameter SRL = 6'b000010;
    parameter SRA = 6'b000011;
    parameter SLLV = 6'b000100;
    parameter SRLV = 6'b000110;
    parameter SRAV = 6'b000111;
    parameter JR = 6'b001000;
    
    parameter LUI = 6'b001111;
      
    
    wire signed [31:0] a_signed;
    wire signed [31:0] b_signed;
    
  
    reg [32:0] res;
    
    
    assign a_signed = a;
    assign b_signed = b;
    assign r = res[31:0];
    
    assign flag = (aluc == SLT || aluc == SLTU) ? ((aluc == SLT) ? (a_signed < b_signed) : (a < b)) : 1'bz; 
    assign zero = (res == 32'b0) ? 1'b1 : 1'b0;
    
    always @ (a or b or aluc)
    begin
        case(aluc)
            ADD: begin
                res <= a_signed + b_signed;
            end
            ADDU: begin
                res <= a + b;
            end
            SUB: begin 
                res <= a_signed - b_signed;
            end
            SUBU: begin 
                res <= a - b;
            end
            AND: begin
                res <= a & b;
            end
            OR: begin
                res <= a | b;
            end
            XOR: begin
                res <= a ^ b;
            end
            NOR: begin
                res <= ~(a | b);
            end
            SLT: begin
                res <= a_signed < b_signed ? 1 : 0;
            end
            SLTU: begin
                res <= a < b ? 1 : 0;
            end
            SLL: begin
                res <= b << a;
            end
            SRL: begin
                res <= b >> a;
            end
            SRA: begin
                res <= b_signed >>> a_signed;
            end
            SLLV: begin
                res <= b << a[4:0];
            end
            SRLV: begin
                res <= b >> a[4:0];
            end
            SRAV: begin
                res <= b_signed >>> a_signed[4:0];
            end
            LUI: begin
                res <= {a[15:0], 16'h0000};
            end
            default:
            begin
                res <= 32'bz;
            end
        endcase
    end
endmodule
"""

BENCHMARKS = ("alu", "multiplier", "shifter", "counter", "arbiter", "fifo")
TOP = dict(zip(BENCHMARKS, ("alu", "multi_8bit", "barrel_shifter", "counter_12", "arbiter", "fifo")))
PORTS = {
    "alu": (("a", "b", "aluc"), ("r", "zero", "carry", "negative", "overflow", "flag"), None),
    "multiplier": (("A", "B"), ("product",), None),
    "shifter": (("in", "ctrl"), ("out",), None),
    "counter": (("rst_n", "valid_count"), ("out",), "clk"),
    "arbiter": (("req",), ("grant",), None),
    "fifo": (("rst", "wr_en", "rd_en", "din"), ("dout", "full", "empty"), "clk"),
}
RTLLM_COMMIT = "51ed553d0ffd32797a1a0a13e051656bf302c81f"
RTLLM_PATHS = {
    "alu": "Miscellaneous/RISC-V/alu/verified_alu.v",
    "multiplier": "Arithmetic/Multiplier/multi_8bit/verified_multi_8bit.v",
    "shifter": "Memory/Shifter/barrel_shifter/verified_barrel_shifter.v",
    "counter": "Control/Counter/counter_12/verified_counter_12.v",
}
SPEC = {
    "alu": (
        "32-bit RTLLM-derived ALU. Keep all ports and opcode parameters/defaults. Preserve the "
        "33-bit expression context and define zero from the entire 33-bit res. Immediate shifts "
        "use all of a (amounts >= 32 need the original oversized behavior); variable shifts use "
        "a[4:0]. SLLV is left, SRLV logical right, and SRAV arithmetic right. LUI uses a[15:0]. "
        "carry, negative and overflow are constant 0. flag is the comparator for SLT/SLTU "
        "and 0 for all other opcodes. Unsupported opcodes, including JR, give r=0 and zero=1."
    ),
    "arbiter": (
        "8-bit combinational, lowest index wins. req=0x00 -> grant=0x00; req=0x01 -> 0x01; "
        "req=0x06 -> 0x02; req=0x81 or 0xff -> 0x01. One-hot grant. "
        "Bit i wins iff req[i] is 1 and no lower numbered bit is 1."
    ),
    "fifo": (
        "Four 8-bit entries; synchronous active-high reset at posedge. Compute take_write "
        "from wr_en and !full, and take_read from rd_en and !empty using pre-edge flags. "
        "If both are accepted, both pointers advance and occupancy holds. If full, a simultaneous "
        "request reads only; if empty, it writes only. dout holds outside valid reads and resets to 0."
    ),
    "shifter": (
        "8-bit zero-fill logical right shift, out = in >> ctrl for all ctrl=0..7. "
        "The bits form a combined shift amount, not a one-hot selector. Example: 8'h80 >> 3 = 8'h10."
    ),
    "multiplier": "Combinational full-width 16-bit unsigned product of 8-bit inputs A and B.",
    "counter": (
        "4-bit modulo-12 counter, asynchronous active-low reset to zero, rising-edge clock, "
        "valid_count enables increments. Wrap 11 to 0; hold when disabled."
    ),
}

def reference_suite():
    refs = dict(DRIVE_SNAPSHOT)
    # Normalize undefined outputs only. Keep the upstream opcode table and 33-bit datapath.
    alu = refs["alu"]
    flag_z, default_z, result = ": 1'bz;", "res <= 32'bz;", "assign r = res[31:0];"
    assert all(alu.count(token) == 1 for token in (flag_z, default_z, result))
    refs["alu"] = (alu.replace(flag_z, ": 1'b0;").replace(default_z, "res <= 33'b0;")
                   .replace(result, result + "\n    assign carry = 1'b0;\n"
                            "    assign negative = 1'b0;\n    assign overflow = 1'b0;"))
    return refs

def save_references():
    directory = OUTPUT / "references"
    directory.mkdir(parents=True, exist_ok=True)
    write_text(directory / "alu_upstream.v", ALU_UPSTREAM)
    provenance = {}
    for name, rtl in REFERENCES.items():
        path = directory / f"{name}.v"
        write_text(path, rtl)
        drive_path = PROJECT / "benchmarks" / name / "reference.v"
        drive_snapshot_hash = digest(DRIVE_SNAPSHOT[name])
        current_hash = digest(drive_path.read_bytes()) if drive_path.is_file() else None
        if current_hash and current_hash != drive_snapshot_hash:
            print(f"{name}: Drive file differs from the project snapshot bundled here.")
        provenance[name] = {
            "top": TOP[name], "reference_sha256": digest(rtl),
            "original_project_reference_sha256": drive_snapshot_hash,
            "current_drive_reference_sha256": current_hash,
            "rtllm_commit": RTLLM_COMMIT if name in RTLLM_PATHS else None,
            "rtllm_source": RTLLM_PATHS.get(name), "contract": SPEC[name],
            "profile": "rtllm-alu-defined-io-v1" if name == "alu" else "project-reference",
        }
    write_json(OUTPUT / "provenance.json", provenance)
    return provenance

REFERENCES = reference_suite()


In [ ]:
TESTBENCHES = {}

TESTBENCHES["arbiter"] = r"""
reg [7:0] req, expected;
wire [7:0] gold, got;
integer i,j;
golden_arbiter g(req,gold);
arbiter d(req,got);
initial begin
  for (i=0;i<256;i=i+1) begin
    req=i; expected=0;
    for(j=0;j<8;j=j+1)
      if(req[j] && expected==0) expected=8'h1 << j;
    #1;
    if(gold !== expected) $fatal(1,"REFERENCE arbiter req=%02h expected=%02h ref=%02h",req,expected,gold);
    if(got !== expected) $fatal(1,"Arbiter req=%02h expected=%02h got=%02h",req,expected,got);
  end
  $display("RTL_SIM_PASS checks=%0d",256); $finish;
end
"""

TESTBENCHES["shifter"] = r"""
reg [7:0] in, expected;
reg [2:0] ctrl;
wire [7:0] gold,got;
integer i,j;
golden_barrel_shifter g(in,ctrl,gold);
barrel_shifter d(in,ctrl,got);
initial begin
  for (i=0;i<256;i=i+1) for(j=0;j<8;j=j+1) begin
    in=i; ctrl=j; expected=i >> j; #1;
    if(gold !== expected) $fatal(1,"REFERENCE shifter in=%02h ctrl=%0d expected=%02h ref=%02h",in,ctrl,expected,gold);
    if(got !== expected) $fatal(1,"Shifter in=%02h ctrl=%0d expected=%02h got=%02h",in,ctrl,expected,got);
  end
  $display("RTL_SIM_PASS checks=%0d",2048); $finish;
end
"""

TESTBENCHES["multiplier"] = r"""
reg [7:0] A,B;
reg [15:0] expected;
wire [15:0] gold,got;
integer i,j;
golden_multi_8bit g(A,B,gold);
multi_8bit d(A,B,got);
initial begin
  for (i=0;i<256;i=i+1) for(j=0;j<256;j=j+1) begin
    A=i; B=j; expected=i*j; #1;
    if(gold !== expected) $fatal(1,"REFERENCE multiplier A=%02h B=%02h expected=%04h ref=%04h",A,B,expected,gold);
    if(got !== expected) $fatal(1,"Multiplier A=%02h B=%02h expected=%04h got=%04h",A,B,expected,got);
  end
  $display("RTL_SIM_PASS checks=%0d",65536); $finish;
end
"""

TESTBENCHES["alu"] = r"""
reg [31:0] a,b;
reg [5:0] aluc;
wire [31:0] rg,rd,ru;
wire zg,cg,ng,og,fg,zd,cd,nd,od,fd,zu,cu,nu,ou,fu;
reg [31:0] corners [0:11];
integer i,j,k,seed,checks;
golden_alu g(a,b,aluc,rg,zg,cg,ng,og,fg);
alu d(a,b,aluc,rd,zd,cd,nd,od,fd);
verified_alu upstream(a,b,aluc,ru,zu,cu,nu,ou,fu);
task check;
  begin
    #1;
    if ((^ru !== 1'bx && rg !== ru) || (zu !== 1'bx && zu !== 1'bz && zg !== zu) ||
        (fu !== 1'bz && fg !== fu) ||
        ((^ru === 1'bx) && {rg,zg} !== {32'b0,1'b1}) ||
        ((fu === 1'bz) && fg !== 1'b0) || {cg,ng,og} !== 3'b0)
      $fatal(1,"REFERENCE ALU op=%02h a=%08h b=%08h normalized=%08h/%b%b upstream=%08h/%b%b",
             aluc,a,b,rg,zg,fg,ru,zu,fu);
    if ({rg,zg,cg,ng,og,fg} !== {rd,zd,cd,nd,od,fd})
      $fatal(1,"ALU op=%02h a=%08h b=%08h expected r=%08h z/c/n/o/f=%b%b%b%b%b got r=%08h z/c/n/o/f=%b%b%b%b%b",
             aluc,a,b,rg,zg,cg,ng,og,fg,rd,zd,cd,nd,od,fd);
    checks=checks+1;
  end
endtask
initial begin
  seed=20261119; checks=0;
  corners[0]=0; corners[1]=1; corners[2]=2; corners[3]=15; corners[4]=31;
  corners[5]=32; corners[6]=33; corners[7]=32'h7fffffff; corners[8]=32'h80000000;
  corners[9]=32'hffffffff; corners[10]=32'h5555aaaa; corners[11]=32'haaaa5555;
  for(i=0;i<64;i=i+1) begin
    aluc=i;
    for(j=0;j<12;j=j+1) for(k=0;k<12;k=k+1) begin a=corners[j]; b=corners[k]; check; end
    for(j=0;j<40;j=j+1) begin a=$random(seed); b=$random(seed); check; end
  end
  $display("RTL_SIM_PASS checks=%0d",checks); $finish;
end
"""

TESTBENCHES["counter"] = r"""
reg clk=0, rst_n=1, valid_count=0;
wire [3:0] gold, got;
reg [3:0] expected;
integer i, checks=0;
golden_counter_12 g(rst_n,clk,valid_count,gold);
counter_12 d(rst_n,clk,valid_count,got);
always #5 clk=~clk;
task check;
  begin
    if(gold !== expected) $fatal(1,"REFERENCE counter cycle=%0d expected=%0d ref=%0d",i,expected,gold);
    if(got !== expected) $fatal(1,"Counter cycle=%0d rst_n=%b valid_count=%b expected=%0d got=%0d",i,rst_n,valid_count,expected,got);
    checks=checks+1;
  end
endtask
initial begin
  i=-1; #1; rst_n=0; expected=0; #1; check;
  @(negedge clk); rst_n=1;
  for (i=0;i<150;i=i+1) begin
    @(negedge clk); valid_count=(i%7!=0);
    if(i%43==32) begin #1; rst_n=0; expected=0; #1; check; rst_n=1; end
    @(posedge clk);
    if(valid_count) expected = expected==11 ? 0 : expected+1;
    #1; check;
  end
  $display("RTL_SIM_PASS checks=%0d",checks); $finish;
end
"""

TESTBENCHES["fifo"] = r"""
reg clk=0, rst=1, wr_en=0, rd_en=0;
reg [7:0] din=0, expected_data;
reg [7:0] queue [0:3];
wire [7:0] gold,got;
wire full_gold,empty_gold,full_got,empty_got;
integer count,checks=0,i,j,seed=20261119;
reg take_rd,take_wr;
always #5 clk=~clk;
golden_fifo g(clk,rst,wr_en,rd_en,din,gold,full_gold,empty_gold);
fifo d(clk,rst,wr_en,rd_en,din,got,full_got,empty_got);
task step(input reset, input wr, input rd, input [7:0] data);
  begin
    @(negedge clk);
    rst=reset; wr_en=wr; rd_en=rd; din=data;
    @(posedge clk);
    if(reset) begin count=0; expected_data=0; end
    else begin
      take_rd = rd && count>0;
      take_wr = wr && count<4;
      if(take_rd) begin
        expected_data=queue[0];
        for(j=0;j<3;j=j+1) queue[j]=queue[j+1];
        count=count-1;
      end
      if(take_wr) begin queue[count]=data; count=count+1; end
    end
    #1;
    if ({gold,full_gold,empty_gold} !== {expected_data,(count==4),(count==0)})
      $fatal(1,"REFERENCE FIFO cycle=%0d count=%0d",checks,count);
    if ({got,full_got,empty_got} !== {expected_data,(count==4),(count==0)})
      $fatal(1,"FIFO cycle=%0d rst=%b wr=%b rd=%b din=%02h expected dout=%02h full=%b empty=%b got dout=%02h full=%b empty=%b",
        checks,rst,wr_en,rd_en,din,expected_data,(count==4),(count==0),got,full_got,empty_got);
    checks=checks+1;
  end
endtask
initial begin
  step(1,0,0,0);
  step(0,1,1,8'h80);
  repeat(4) step(0,1,1,8'h81 + checks);
  for(i=0;i<6;i=i+1) step(0,1,0,8'ha0 + i);
  step(0,1,1,8'h55);
  repeat(6) step(0,0,1,0);
  for(i=0;i<300;i=i+1) step(i%57==0,$random(seed),$random(seed),$random(seed));
  $display("RTL_SIM_PASS checks=%0d",checks); $finish;
end
"""

def write_candidate(work, rtl):
    work.mkdir(parents=True, exist_ok=True)
    (work / "candidate.v").write_text(rtl)

def simulation_check(name, rtl, work):
    work.mkdir(parents=True, exist_ok=True)
    source = REFERENCES[name]
    names = set(re.findall(r"(?m)^\s*module\s+(\w+)", source))
    golden = re.sub(r"\b(?:" + "|".join(sorted(names)) + r")\b",
                    lambda match: "golden_" + match.group(), source)
    (work / "golden.v").write_text(golden)
    write_candidate(work, rtl)
    (work / "tb.v").write_text("\x60timescale 1ns/1ps\nmodule tb;\n" + TESTBENCHES[name] + "endmodule\n")
    sources = ["golden.v", "candidate.v", "tb.v"]
    if name == "alu":
        (work / "upstream.v").write_text(ALU_UPSTREAM)
        sources.insert(0, "upstream.v")
    compile_result = command(["iverilog", "-g2012", "-Wall", "-s", "tb", "-o", "sim.out", *sources],
                             cwd=work, timeout=45, log=work / "compile.log")
    result = {"compile": "PASS", "simulation": "NOT_RUN", "checks": None,
              "simulation_seconds": None, "diagnostic": ""}
    if compile_result["status"] != "DONE" or compile_result["rc"] != 0:
        result.update(compile=compile_result["status"] if compile_result["status"] != "DONE" else "FAIL",
                      diagnostic=compile_result["text"][-3500:])
        return result
    run = command(["vvp", "sim.out"], cwd=work, timeout=SIM_TIMEOUT_S, log=work / "simulation.log")
    marker = re.search(r"(?m)^RTL_SIM_PASS checks=([1-9]\d*)\s*$", run["text"])
    if run["status"] != "DONE":
        status = run["status"]
    elif "FATAL:" in run["text"] and "REFERENCE " in run["text"]:
        status = "REFERENCE_ERROR"
    elif run["rc"] == 0 and marker:
        status = "PASS"
    elif re.search(r"(?m)^FATAL:", run["text"]):
        status = "MISMATCH"
    else:
        status = "TOOL_ERROR" if run["rc"] else "INCOMPLETE"
    result.update(simulation=status, simulation_seconds=run["seconds"],
                  checks=int(marker[1]) if status=="PASS" else None,
                  diagnostic=run["text"][-3500:] if status!="PASS" else "")
    return result


In [ ]:
def check_interface(name, rtl, work):
    write_candidate(work, rtl)
    script = (f"read_verilog -sv candidate.v; hierarchy -check -top {TOP[name]}; "
              "proc; check -assert; write_json interface.json")
    run = command(["yosys", "-Q", "-T", "-p", script], cwd=work, timeout=45,
                  log=work / "interface.log")
    if run["status"] != "DONE" or run["rc"] != 0:
        return None, run["text"][-2500:]
    module = json.loads((work / "interface.json").read_text())["modules"][TOP[name]]
    fields = ("direction", "offset", "upto", "signed")
    ports = {port: [data["direction"], len(data["bits"]),
                   *(data.get(field, 0) for field in fields[1:])]
             for port, data in module["ports"].items()}
    return {"ports": ports, "port_order": list(ports),
            "parameters": module.get("parameter_default_values", {})}, ""

FORMAL_WRAPPERS = {
    "fifo": """module rtl_reset_contract(input clk,rst,wr_en,rd_en, input [7:0] din,
    output [7:0] dout, output full,empty);
    reg boot = 1'b1;
    wire [7:0] data;
    wire f,e;
    always @(posedge clk) boot <= 1'b0;
    fifo dut(.clk(clk),.rst(rst | boot),.wr_en(wr_en),.rd_en(rd_en),.din(din),
             .dout(data),.full(f),.empty(e));
    assign {dout,full,empty} = boot ? 10'b0 : {data,f,e};
endmodule
""",
    "counter": """module rtl_reset_contract(input rst_n,clk,valid_count, output [3:0] out);
    reg boot = 1'b1;
    wire [3:0] data;
    always @(posedge clk) boot <= 1'b0;
    counter_12 dut(.rst_n(rst_n & ~boot),.clk(clk),.valid_count(valid_count),.out(data));
    assign out = boot ? 4'b0 : data;
endmodule
""",
}

def proof_check(name, rtl, work):
    work.mkdir(parents=True, exist_ok=True)
    write_candidate(work, rtl)
    (work / "gold.v").write_text(REFERENCES[name])
    prep_top = TOP[name]
    prep_extra = ""
    collect = ""
    if name in FORMAL_WRAPPERS:
        (work / "reset_contract.v").write_text(FORMAL_WRAPPERS[name])
        prep_top = "rtl_reset_contract"
        prep_extra = "read_verilog -sv reset_contract.v\n"
        if name == "fifo":
            collect = "[collect rtl_reset_contract]\ngroup /dout|full|empty/\n"
    prep = f"{prep_extra}prep -top {prep_top} -flatten\nmemory_map\nopt_clean\nrename -hide"
    cfg = f"""[gold]
read_verilog -sv gold.v
{prep}
[gate]
read_verilog -sv candidate.v
{prep}
{collect}[strategy quick]
use sat
depth 12
[strategy pdr]
use sby
engine abc pdr
timeout {max(20, FORMAL_TIMEOUT_S - 15)}
"""
    (work / "compare.eqy").write_text(cfg)
    log = work / "eqy.log"
    run = command(["eqy", "-f", "-d", "proof", "-j", str(FORMAL_WORKERS), "compare.eqy"],
                  cwd=work, timeout=FORMAL_TIMEOUT_S, log=log)
    lines = run["text"]
    traces = sorted((work / "proof").rglob("*.vcd"))
    done = re.findall(r"DONE\s*\(([A-Z]+)", lines)
    verdict = done[-1] if done else ""
    if run["status"] != "DONE":
        status = run["status"]
    elif run["rc"] == 0 and verdict == "PASS":
        status = "PASS"
    elif verdict == "FAIL" and traces:
        status = "INEQUIVALENT"
    elif verdict == "TIMEOUT" or re.search(r"returned (?:TIMEOUT|timeout)|DONE \(TIMEOUT", lines):
        status = "TIMEOUT"
    elif verdict in {"FAIL", "UNKNOWN"}:
        status = "UNPROVEN"
    else:
        status = "TOOL_ERROR"
    trace_file = ""
    hint = lines[-2500:]
    trace_candidates = traces[:10] if status == "INEQUIVALENT" else []
    for number, source in enumerate(trace_candidates, 1):
        target = work / f"counterexample_{number:02d}.vcd"
        shutil.copyfile(source, target)
        if number == 1:
            trace_file = target.name
            hint += "\n" + compact_vcd(target)
    return {"formal": status, "formal_seconds": run["seconds"],
            "counterexample": trace_file, "formal_diagnostic": hint if status != "PASS" else ""}

def compact_vcd(path):
    symbols = {}
    samples = collections.defaultdict(list)
    stamp = "0"
    for line in path.read_text(errors="replace").splitlines():
        fields = line.split()
        if not fields:
            continue
        if fields[0] == "$var" and len(fields) >= 6:
            symbols[fields[3]] = fields[4]
        elif line.startswith("#"):
            stamp = line[1:]
        elif line[0] in "01xz" and line[1:] in symbols:
            samples[line[1:]].append(f"{stamp}:{line[0]}")
        elif line[0] in "bB" and len(fields) == 2 and fields[1] in symbols:
            samples[fields[1]].append(f"{stamp}:{fields[0][1:]}")
    return "\n".join(f"{symbols[key]} {', '.join(values[:8])}"
                     for key, values in list(samples.items())[:20])[:2200]


In [ ]:
NUMBER = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"

def measurement_config():
    hashes = {key: digest(path.read_bytes()) for key, path in TECHNOLOGY.items()}
    liberty_text = TECHNOLOGY["liberty"].read_text(errors="replace")
    time_unit = re.search(r'time_unit\s*:\s*"\s*1\s*ns\s*"', liberty_text, re.I)
    cap_unit = re.search(r"capacitive_load_unit\s*\(\s*(" + NUMBER + r")\s*,\s*(ff|pf)\s*\)",
                         liberty_text, re.I)
    if not (time_unit and cap_unit):
        raise ValueError("Expected the Nangate45 Liberty with ns timing and fF or pF load units.")
    scale = float(cap_unit[1]) * (0.001 if cap_unit[2].lower() == "ff" else 1)
    if scale <= 0:
        raise ValueError("Invalid Liberty capacitance unit")
    return {
        "schema": 4, "technology_sha256": hashes,
        "baseline_sta": "standalone OpenSTA" if STA_EXEC else "OpenROAD/OpenSTA",
        "clock_ns": CLOCK_NS, "input_slew_ns": INPUT_SLEW_NS, "output_load_pf": OUTPUT_LOAD_PF,
        "output_load_in_liberty_units": OUTPUT_LOAD_PF / scale,
        "area_limit_factor": MAX_AREA_FACTOR, "delay_limit_factor": MAX_DELAY_FACTOR,
        "tool_versions": software_versions(),
        "benchmarks": {name: {"reference": digest(rtl), "testbench": digest(TESTBENCHES[name])}
                       for name, rtl in REFERENCES.items()},
    }

def mapped_synthesis(name, rtl, work):
    write_candidate(work, rtl)
    lib = TECHNOLOGY["liberty"]
    script = (f'read_verilog -sv candidate.v; synth -top {TOP[name]} -flatten -noabc; '
              f'dfflibmap -liberty "{lib}"; abc -liberty "{lib}"; clean -purge; '
              f'stat -liberty "{lib}"; write_json mapped.json; write_verilog -noattr mapped.v')
    (work / "synthesis.ys").write_text(script + "\n")
    proc = command(["yosys", "-Q", "-T", "-s", "synthesis.ys"], cwd=work,
                   timeout=160, log=work / "yosys_synth.log")
    row = {"synth_status": "ERROR", "synth_seconds": proc["seconds"], "area_um2": None,
           "cell_count": None, "cell_mix": {}, "diagnostic": proc["text"][-2500:]}
    if proc["status"] != "DONE" or proc["rc"] != 0:
        row["synth_status"] = proc["status"] if proc["status"] != "DONE" else "ERROR"
        return row
    areas = re.findall(r"Chip area for (?:top )?module[^\r\n]*?:\s*(" + NUMBER + r")",
                       proc["text"], re.I)
    if not areas or not (work / "mapped.json").is_file() or not (work / "mapped.v").is_file():
        row["synth_status"] = "METRIC_MISSING"
        return row
    top = json.loads((work / "mapped.json").read_text())["modules"].get(TOP[name], {})
    mix = dict(collections.Counter(cell["type"] for cell in top.get("cells", {}).values()))
    if not mix or any(cell.startswith("$") for cell in mix):
        row.update(synth_status="UNMAPPED_OR_EMPTY", diagnostic=str(mix))
        return row
    area = float(areas[-1])
    if not math.isfinite(area) or area <= 0:
        row["synth_status"] = "INVALID_AREA"
        return row
    row.update(synth_status="PASS", area_um2=area, cell_count=sum(mix.values()),
               cell_mix=mix, diagnostic="")
    write_json(work / "cell_mix.json", mix)
    return row

def timing_analysis(name, work):
    ins, outs, clock = PORTS[name]
    clock_cmd = f"create_clock -name clk -period {CLOCK_NS}"
    if clock:
        clock_cmd += f" [get_ports {{{clock}}}]"
    commands = []
    if not STA_EXEC:
        commands.extend([f'read_lef {{{TECHNOLOGY["tech_lef"]}}}',
                         f'read_lef {{{TECHNOLOGY["cells_lef"]}}}'])
    commands.extend([
        f'read_liberty {{{TECHNOLOGY["liberty"]}}}', 'read_verilog mapped.v',
        f'link_design {TOP[name]}', clock_cmd,
        f'if {{[llength [get_ports {{{" ".join(ins)}}}]] == 0}} {{error "missing input ports"}}',
        f'set_input_delay -clock clk 0 [get_ports {{{" ".join(ins)}}}]',
        f'set_output_delay -clock clk 0 [get_ports {{{" ".join(outs)}}}]',
        f'set_input_transition {INPUT_SLEW_NS} [get_ports {{{" ".join(ins)}}}]',
        f'set_load {MEASUREMENT["output_load_in_liberty_units"]} [get_ports {{{" ".join(outs)}}}]',
        'puts "RTL_STA_PATHS"',
        'report_checks -path_delay max -digits 6',
        'puts "RTL_STA_DONE"',
    ])
    script = "if {[catch {\n  " + "\n  ".join(commands) + \
             '\n} message]} {puts stderr "RTL_STA_ERROR $message"; exit 1}\n'
    (work / "timing.tcl").write_text(script)
    argv = [STA_EXEC, "-exit", "timing.tcl"] if STA_EXEC else [OPENROAD_EXEC, "-exit", "timing.tcl"]
    proc = command(argv, cwd=work, timeout=120, log=work / "sta_report.log")
    row = {"timing_status": "ERROR", "sta_seconds": proc["seconds"], "delay_ns": None,
           "slack_ns": None, "diagnostic": proc["text"][-2500:]}
    if proc["status"] != "DONE" or proc["rc"] != 0:
        row["timing_status"] = proc["status"] if proc["status"] != "DONE" else "ERROR"
        return row
    report = proc["text"]
    if "RTL_STA_DONE" not in report or re.search(r"(?im)^\s*(?:\[\s*)?(?:error:|error\s)", report):
        return row
    paths = report.split("RTL_STA_PATHS")[-1].split("RTL_STA_DONE")[0]
    found = re.findall(r"(?m)^\s*(" + NUMBER + r")\s+slack\s+\((?:MET|VIOLATED)\)", paths)
    if not found or "Startpoint:" not in paths or "Endpoint:" not in paths:
        row["timing_status"] = "UNCONSTRAINED_OR_MISSING_PATH"
        return row
    slack = min(map(float, found))
    delay = CLOCK_NS - slack
    if not math.isfinite(delay) or delay <= 0:
        row["timing_status"] = "INVALID_DELAY"
        return row
    row.update(timing_status="PASS", delay_ns=round(delay, 6), slack_ns=slack, diagnostic="")
    return row

def measure(name, rtl, work):
    row = mapped_synthesis(name, rtl, work)
    if row["synth_status"] == "PASS":
        row.update(timing_analysis(name, work))
    else:
        row.update(timing_status="NOT_RUN", sta_seconds=None, delay_ns=None, slack_ns=None)
    row["adp"] = round(row["area_um2"] * row["delay_ns"], 9) if row["timing_status"] == "PASS" else None
    write_json(work / "ppa_metrics.json", row)
    return row


In [ ]:
import pandas as pd

FIELDS = (
    "utc", "attempt_id", "experiment_id", "run", "benchmark", "workflow", "model",
    "model_id", "method", "attempt", "seed", "config_id", "source_sha256", "candidate_path",
    "artifact_dir", "action", "policy_state", "reward", "status", "error", "syntax", "compile",
    "interface", "simulation", "checks", "formal", "formal_seconds", "counterexample_path",
    "generation_seconds", "input_tokens", "output_tokens", "simulation_seconds", "synth_status",
    "verification_reused_from",
    "timing_status", "synth_seconds", "sta_seconds", "cell_count", "cell_mix",
    "area_um2", "delay_ns", "slack_ns", "adp", "baseline_area_um2", "baseline_delay_ns",
    "baseline_adp", "area_improvement_pct", "delay_improvement_pct", "adp_improvement_pct",
    "adp_improved", "eligible_under_limits", "new_best", "best_allowed_adp", "audit_status",
    "audit_seconds", "audit_diagnostic", "assurance", "competition_eligible",
)

def utc():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def save_csv(path, table):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(tmp, index=False)
    if path.is_file() and digest(path.read_bytes()) == digest(tmp.read_bytes()):
        tmp.unlink()
    else:
        os.replace(tmp, path)

def archive(work, dest):
    for file in work.rglob("*"):
        if not file.is_file():
            continue
        relative = file.relative_to(work)
        if "proof" in relative.parts:
            if file.name not in {"logfile.txt", "status", "partition.ids"}:
                continue
        elif file.name == "interface.json" or file.suffix not in {".v", ".log", ".txt", ".eqy", ".vcd", ".tcl", ".ys", ".json"}:
            continue
        target = dest / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(file, target)

def stage(location, fn):
    with tempfile.TemporaryDirectory(prefix="rtl-", dir=RUNTIME) as directory:
        work = Path(directory)
        try:
            return fn(work)
        finally:
            archive(work, location)

def init_row(name, method, run, attempt, dest, model, model_id, seed):
    base = BASELINES[name]
    dest.mkdir(parents=True, exist_ok=False)
    row = dict.fromkeys(FIELDS)
    row.update(utc=utc(), attempt_id=f"{run.name}/{dest.name}", experiment_id=run.name,
               run=run.label, benchmark=name, workflow=WORKFLOW, model=model, model_id=model_id,
               method=method, attempt=attempt, seed=seed, config_id=CONFIG_ID,
               candidate_path="", artifact_dir=str(dest), status="STARTED", error="",
               syntax="NOT_RUN", compile="NOT_RUN", interface="NOT_RUN", simulation="NOT_RUN",
               formal="NOT_RUN", audit_status="NOT_RUN", synth_status="NOT_RUN", timing_status="NOT_RUN",
               adp_improved=False, eligible_under_limits=False, new_best=False,
               baseline_adp=base["adp"], baseline_area_um2=base["area_um2"],
               baseline_delay_ns=base["delay_ns"])
    write_json(dest / "record.json", row)
    return row

def finish_row(row):
    write_json(Path(row["artifact_dir"]) / "record.json", row)
    return row

def candidate_check(row, rtl):
    name = row["benchmark"]
    dest = Path(row["artifact_dir"])
    write_candidate(dest, rtl)
    row.update(candidate_path=str(dest / "candidate.v"), source_sha256=digest(rtl))

    sim = stage(dest / "simulation", lambda work: simulation_check(name, rtl, work))
    row.update({key: sim[key] for key in ("compile", "simulation", "simulation_seconds", "checks")})
    row["syntax"] = sim["compile"]
    if sim["simulation"] != "PASS":
        row.update(status=(
            f"COMPILE_{sim['compile']}" if sim["compile"] != "PASS"
            else f"SIMULATION_{sim['simulation']}"
        ), error=sim["diagnostic"])
        return row
    signature, error = stage(dest / "interface", lambda work: check_interface(name, rtl, work))
    if signature != BASELINES[name]["interface_signature"]:
        row.update(interface="ERROR" if not signature else "MISMATCH", status="INTERFACE_FAIL",
                   error=error or f"Expected {BASELINES[name]['interface_signature']}; got {signature}")
        return row
    row["interface"] = "PASS"

    if row["method"] in {"formal", "formal_ppa", "q_learning", "random_policy"}:
        proof = stage(dest / "formal", lambda work: proof_check(name, rtl, work))
        row.update(formal=proof["formal"], formal_seconds=proof["formal_seconds"])
        if proof["counterexample"]:
            row["counterexample_path"] = str(dest / "formal" / proof["counterexample"])
        if row["formal"] != "PASS":
            row.update(status=f"FORMAL_{row['formal']}", error=proof["formal_diagnostic"])
            return row

    metrics = stage(dest / "ppa", lambda work: measure(name, rtl, work))
    row.update({key: metrics[key] for key in (
        "synth_status", "timing_status", "synth_seconds", "sta_seconds", "cell_mix",
        "cell_count", "area_um2", "delay_ns", "slack_ns", "adp")})
    if row["adp"] is None:
        row.update(status=f"PPA_{metrics['synth_status']}_{metrics['timing_status']}",
                   error=metrics["diagnostic"])
        return row
    for metric, field in [("area_um2", "area"), ("delay_ns", "delay"), ("adp", "adp")]:
        before = BASELINES[name][metric]
        row[f"{field}_improvement_pct"] = 100.0 * (before - row[metric]) / before
    row["adp_improved"] = row["adp"] < row["baseline_adp"] - 1e-9
    row["eligible_under_limits"] = (
        row["area_um2"] <= MAX_AREA_FACTOR * row["baseline_area_um2"]
        and row["delay_ns"] <= MAX_DELAY_FACTOR * row["baseline_delay_ns"])
    row["status"] = "MEASURED"
    return row

def read_json(path):
    return json.loads(Path(path).read_text())

def all_attempts(directory=OUTPUT):
    rows = []
    for path in sorted(directory.glob("runs/*/attempt*/record.json")):
        try:
            row = read_json(path)
        except (OSError, ValueError) as exc:
            print(f"Could not read {path}: {exc}")
            continue
        if row["status"] == "STARTED":
            row.update(status="INTERRUPTED", error="Attempt ended before a final result was saved.")
            saved_candidate = path.parent / "candidate.v"
            if saved_candidate.is_file():
                row.update(candidate_path=str(saved_candidate), source_sha256=digest(saved_candidate.read_bytes()))
        audit_path = path.parent / "audit.json"
        if audit_path.is_file():
            audit = read_json(audit_path)
            row.update(audit_status=audit["formal"], audit_seconds=audit.get("formal_seconds"),
                       audit_diagnostic=audit.get("formal_diagnostic", ""))
            if audit.get("counterexample_path"):
                row["counterexample_path"] = audit["counterexample_path"]
        if row["formal"] == "PASS" or row["audit_status"] == "PASS":
            row["assurance"] = "FORMALLY_VERIFIED"
        elif row["audit_status"] == "INEQUIVALENT" or row["formal"] == "INEQUIVALENT":
            row["assurance"] = "INEQUIVALENT"
        elif row["audit_status"] != "NOT_RUN":
            row["assurance"] = f"AUDIT_{row['audit_status']}"
        elif row["simulation"] == "PASS" and row["formal"] == "NOT_RUN":
            row["assurance"] = "SIMULATION_ONLY"
        else:
            row["assurance"] = "UNVERIFIED"
        row["competition_eligible"] = bool(row["assurance"] == "FORMALLY_VERIFIED"
                                            and row["eligible_under_limits"] and row["adp_improved"])
        rows.append(row)
    return pd.DataFrame(rows, columns=FIELDS).sort_values(
        ["utc", "attempt_id"], ignore_index=True)

def audit_simulation_passes(config_id=None, improvements_only=False):
    config_id = CONFIG_ID if config_id is None else config_id
    table = all_attempts()
    eligible = (table["config_id"].eq(config_id) & table["adp"].notna()
                & table["formal"].eq("NOT_RUN") & table["audit_status"].eq("NOT_RUN"))
    if improvements_only:
        eligible &= table["adp_improved"].eq(True)
    for row in table.loc[eligible].to_dict("records"):
        dest = Path(row["artifact_dir"])
        candidate = Path(row["candidate_path"])
        try:
            rtl = candidate.read_text()
            if digest(rtl) != row["source_sha256"]:
                raise ValueError("Candidate RTL hash has changed.")
            proof = stage(dest / "audit", lambda work: proof_check(row["benchmark"], rtl, work))
        except (OSError, ValueError) as exc:
            proof = {"formal": "TOOL_ERROR", "formal_diagnostic": str(exc)}
        if proof.get("counterexample"):
            proof["counterexample_path"] = str(dest / "audit" / proof["counterexample"])
        write_json(dest / "audit.json", {**proof, "utc": utc(), "attempt_id": row["attempt_id"]})
    return export_results()

def export_results(directory=OUTPUT):
    table = all_attempts(directory)
    passed = table.loc[table["simulation"].eq("PASS")].copy()
    failure = table.loc[~table["status"].isin({"MEASURED", "NO_CHANGE"}) | (
        ~table["audit_status"].isin({"PASS", "NOT_RUN"}))].copy()
    failure["failure_stage"] = failure["status"].where(failure["status"].ne("MEASURED"),
                                                       "AUDIT_" + failure["audit_status"].fillna("NOT_RUN"))
    improved = table.loc[table["adp_improved"].eq(True)].copy()
    improved["saved_rtl"] = ""
    for index, row in improved.iterrows():
        dest = directory / "improved_rtl" / row["benchmark"] / row["attempt_id"] / "candidate.v"
        src = Path(row["candidate_path"])
        if src.is_file() and digest(src.read_bytes()) == row["source_sha256"]:
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.is_file() or digest(dest.read_bytes()) != row["source_sha256"]:
                shutil.copyfile(src, dest)
            improved.at[index, "saved_rtl"] = str(dest)
    verified = improved.loc[improved["competition_eligible"].eq(True)].copy()
    grouping = ["config_id", "model", "method", "benchmark"]
    best = verified.sort_values(["adp", "attempt_id"]).drop_duplicates(grouping)
    audits = table.loc[~table["audit_status"].eq("NOT_RUN")].copy()
    for filename, frame in {
        "attempts.csv": table, "failures.csv": failure, "simulation_passes.csv": passed,
        "no_change.csv": table.loc[table["status"].eq("NO_CHANGE")],
        "adp_improvements.csv": improved, "verified_improvements.csv": verified,
        "best_verified.csv": best, "formal_audits.csv": audits,
    }.items():
        save_csv(directory / filename, frame)
    return table

def summarize(table, by_run=False):
    groups = ["config_id", "workflow", "model", "method", "benchmark"]
    if by_run:
        groups += ["run", "experiment_id"]
    numeric = table.assign(
        sim_pass=table["simulation"].eq("PASS"),
        proven=table["assurance"].eq("FORMALLY_VERIFIED"),
        verified_gain=table["adp_improvement_pct"].where(table["competition_eligible"].eq(True)),
        elapsed_s=table[["generation_seconds", "simulation_seconds", "formal_seconds",
                         "synth_seconds", "sta_seconds"]].apply(pd.to_numeric, errors="coerce").sum(axis=1),
    )
    return numeric.groupby(groups, dropna=False).agg(
        attempts=("attempt_id", "size"), sim_passes=("sim_pass", "sum"), proven=("proven", "sum"),
        sim_pass_pct=("sim_pass", "mean"), proof_pct=("proven", "mean"),
        best_verified_adp_gain_pct=("verified_gain", "max"), mean_seconds=("elapsed_s", "mean"),
        mean_generation_s=("generation_seconds", "mean"),
        total_input_tokens=("input_tokens", "sum"), total_output_tokens=("output_tokens", "sum"),
    ).reset_index().assign(sim_pass_pct=lambda df: df["sim_pass_pct"]*100,
                          proof_pct=lambda df: df["proof_pct"]*100)

def legacy_attempts(path):
    table = pd.read_csv(path)
    if {"module", "simulation", "status"}.issubset(table.columns):
        table = table.rename(columns={"module": "benchmark", "method": "mode"})
        table["log_status"] = table["simulation"]
        table["path"] = table["candidate_path"].fillna("").map(
            lambda s: str(Path(s).parent) if s else "")
    elif {"benchmark", "mode", "status", "path"}.issubset(table.columns):
        table["log_status"] = table["status"]
    else:
        raise ValueError(f"{Path(path).name} is not an attempt-level report. "
                         "Use attempts_v2.csv or attempt_failure_summary.csv.")
    return table

def legacy_failure_report(benchmark=None, max_logs=4):
    directory = LEGACY_PROJECT / "results"
    summary = directory / "attempt_failure_summary.csv"
    source = summary if summary.is_file() else directory / "attempts_v2.csv"
    if not source.is_file():
        print("No legacy per-attempt report found.")
        return pd.DataFrame()
    rows = legacy_attempts(source)
    if benchmark:
        subset = rows["benchmark"].eq(benchmark)
        failed = ~rows["log_status"].isin({"PASS", "simulation_passed"})
        for item in rows[subset & failed].tail(max_logs).itertuples():
            print(f"\n{item.model} {item.benchmark} {item.mode} {item.log_status}")
            if item.path:
                for filename in ("compile.log", "simulation.log"):
                    primary = Path(item.path) / "simulation" / filename
                    fallback = Path(item.path) / filename
                    path = primary if primary.is_file() else fallback
                    if path.is_file():
                        print(filename, path.read_text(errors="replace")[-1200:])
    return pd.crosstab([rows["model"], rows["benchmark"]], rows["log_status"])

class Experiment:
    def __init__(self, name, method, label, **metadata):
        ready = BASELINES[name]
        if ready["status"] != "READY":
            raise ValueError(f"Baseline {name} is {ready['status']}. Inspect baseline_checks.csv.")
        self.label = str(label)
        settings = {
            "run": self.label, "benchmark": name, "method": method, "workflow": WORKFLOW,
            "config_id": CONFIG_ID, "run_tag": RUN_TAG, "checkpoint_version": 1, **metadata,
        }
        self.name = digest(json.dumps(settings, sort_keys=True))[:20]
        self.directory = OUTPUT / "runs" / self.name
        self.directory.mkdir(parents=True, exist_ok=True)
        manifest = self.directory / "manifest.json"
        if not manifest.is_file():
            write_json(manifest, {"utc": utc(), "experiment_id": self.name,
                                  "measurement": MEASUREMENT, **settings})

    def attempt_dir(self, number):
        return self.directory / f"attempt{number:04d}"

    def committed(self, *, resumable_rl=False):
        """Read the consecutive durable attempts before the first interrupted slot."""
        rows = []
        while (dest := self.attempt_dir(len(rows)+1)).is_dir():
            path = dest / "record.json"
            row = read_json(path) if path.is_file() else None
            if row is None or row["status"] in {"STARTED", "GENERATION_ERROR", "MODEL_PRIOR_ERROR"}:
                break
            if resumable_rl and "_resume" not in row:
                raise RuntimeError(f"Missing RL state in {path}.")
            if resumable_rl and rows:
                rows[-1].pop("_resume")
            rows.append(row)
        return rows

    def retry_slot(self, dest):
        if dest.exists():
            archive_dir = self.directory / "interrupted"
            archive_dir.mkdir(exist_ok=True)
            dest.rename(archive_dir / f"{dest.name}-{uuid.uuid4().hex[:10]}")

    def complete(self):
        marker = self.directory / "complete.json"
        if not marker.is_file():
            write_json(marker, {"utc": utc()})

    @property
    def is_complete(self):
        return (self.directory / "complete.json").is_file()

    def restore_best(self, rows):
        best = next((row for row in reversed(rows) if row["new_best"]), None)
        if best:
            self.save_best(best)
        return best

    def save_best(self, row):
        write_text(self.directory / "best_candidate.v", Path(row["candidate_path"]).read_text())
        label = "FORMALLY_VERIFIED" if row["formal"] == "PASS" else "SIMULATION_ONLY_AT_SELECTION"
        write_json(self.directory / "best_summary.json", {
            **{key: value for key, value in row.items() if key != "_resume"},
            "selection_assurance": label,
        })


def record_best(run, row, incumbent):
    best_adp = incumbent["adp"] if incumbent else row["baseline_adp"]
    if row["status"] == "MEASURED" and row["eligible_under_limits"] and row["adp"] < best_adp - 1e-9:
        row["new_best"] = True
        incumbent = dict(row)

    row["best_allowed_adp"] = incumbent["adp"] if incumbent else row["baseline_adp"]
    return incumbent


In [ ]:
def baseline_check(name, rtl, dest):
    def evaluate(work):
        signature, interface_error = check_interface(name, rtl, work / "interface")
        sim = simulation_check(name, rtl, work / "simulation")
        proof = (proof_check(name, rtl, work / "formal")
                 if signature and sim["simulation"] == "PASS" else {"formal": "NOT_RUN"})
        ppa = (measure(name, rtl, work / "ppa") if proof["formal"] == "PASS"
               else {"area_um2": None, "delay_ns": None, "adp": None,
                     "synth_status": "NOT_RUN", "timing_status": "NOT_RUN"})
        sim_error = sim.get("diagnostic")
        return {**sim, **proof, **ppa, "interface_signature": signature,
                "diagnostic": interface_error or sim_error or proof.get("formal_diagnostic")
                or ppa.get("diagnostic", ""),
                "status": "READY" if ppa["adp"] is not None else "REFERENCE_BLOCKED"}
    return stage(dest, evaluate)

def run_preflight():
    folder = OUTPUT / "baselines" / CONFIG_ID
    folder.mkdir(parents=True, exist_ok=True)
    cache = folder / "baseline_metrics.json"
    cached = read_json(cache) if cache.is_file() else {}
    result = dict(cached)
    for name, rtl in REFERENCES.items():
        row = cached.get(name)
        if row and row["status"] == "READY":
            result[name] = row
        else:
            result[name] = baseline_check(name, rtl, folder / name)
            write_json(cache, result)
        checked_result = result[name]
        print(f"{name:11} {checked_result['status']:18} sim={checked_result['simulation']} "
              f"formal={checked_result['formal']} ADP={checked_result['adp']}")
    write_json(cache, result)
    save_csv(OUTPUT / "baseline_checks.csv", pd.DataFrame.from_dict(result, orient="index")
             .rename_axis("benchmark").reset_index())
    fault = REFERENCES["arbiter"].replace("i = 7; i >= 0; i = i - 1", "i = 0; i < 8; i = i + 1")
    assert fault != REFERENCES["arbiter"]
    negative = folder / "negative_control"
    negative_result = negative / "result.json"
    control = read_json(negative_result) if negative_result.is_file() else {}
    sim, proof = control.get("simulation", {}), control.get("formal", {})
    if sim.get("simulation") != "MISMATCH" or proof.get("formal") != "INEQUIVALENT":
        sim = stage(negative / "simulation", lambda work: simulation_check("arbiter", fault, work))
        proof = stage(negative / "formal", lambda work: proof_check("arbiter", fault, work))
        write_json(negative_result, {"simulation": sim, "formal": proof})
    if sim["simulation"] != "MISMATCH" or proof["formal"] != "INEQUIVALENT":
        raise RuntimeError("Preflight fault injection failed. Inspect the arbiter negative-control logs.")
    if not any(row["status"] == "READY" for row in result.values()):
        raise RuntimeError("No reference passed preflight. Inspect baseline_checks.csv.")
    return result

STA_EXEC, OPENROAD_EXEC = configure_tools()
TECHNOLOGY = configure_technology(not STA_EXEC)
PROVENANCE = save_references()
MEASUREMENT = measurement_config()
MEASUREMENT["verification_contract"] = digest("".join(FORMAL_WRAPPERS.values()) + "EQY ports only v1")
MEASUREMENT["alu_upstream_sha256"] = digest(ALU_UPSTREAM)
CONFIG_ID = digest(json.dumps(MEASUREMENT, sort_keys=True))[:16]
write_json(OUTPUT / "baselines" / CONFIG_ID / "measurement.json", MEASUREMENT)
BASELINES = run_preflight()
READY_BENCHMARKS = tuple(name for name, row in BASELINES.items() if row["status"] == "READY")
print("Outputs:", OUTPUT)


In [ ]:
MODELS = {
    "coder_7b": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "qwen3_8b": "Qwen/Qwen3-8B",
    "coder_14b": "Qwen/Qwen2.5-Coder-14B-Instruct",
    "codev_r1": "zhuyaoyu/CodeV-R1-RL-Qwen-7B",
}
MAX_NEW_TOKENS = 2400

def load_llm(model_key, revision=None):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU in Colab Runtime > Change runtime type.")
    model_id = MODELS[model_key]
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map={"": 0},
                                               torch_dtype=dtype, quantization_config=quant,
                                               use_safetensors=True, revision=revision, trust_remote_code=False)
    model.eval()
    return tokenizer, model

def load_project_model(model_key):
    path = OUTPUT / f"model_{model_key}.json"
    previous = read_json(path) if path.is_file() else {}
    tokenizer, model = load_llm(model_key, revision=previous.get("revision"))
    revision = getattr(model.config, "_commit_hash", None) or previous.get("revision")
    write_json(path, {"id": MODELS[model_key], "revision": revision})
    return tokenizer, model

def model_inputs(prompt, model_key, tokenizer, model):
    messages = [
        {"role": "system", "content": "You optimize synthesizable RTL under an exact fixed hardware contract."},
        {"role": "user", "content": prompt},
    ]
    options = {"enable_thinking": False} if model_key == "qwen3_8b" else {}
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **options)
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
    device = model.get_input_embeddings().weight.device
    return {key: tensor.to(device) for key, tensor in inputs.items()}

def generate_hf(prompt, seed, model_key, tokenizer, model):
    import torch
    torch.manual_seed(seed)
    inputs = model_inputs(prompt, model_key, tokenizer, model)
    prompt_length = inputs["input_ids"].shape[-1]
    start = time.monotonic()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                temperature=0.7, top_p=0.85, top_k=30, use_cache=True,
                                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    completion = output[0, prompt_length:]
    return tokenizer.decode(completion, skip_special_tokens=True), {
        "input_tokens": prompt_length, "output_tokens": int(completion.numel()),
        "generation_seconds": round(time.monotonic() - start, 4),
    }



In [ ]:
def arbiter_prefix():
    equations = [f"  assign grant[{i}] = req[{i}]" + (f" & ~(|req[{i-1}:0]);" if i else ";")
                 for i in range(8)]
    return "module arbiter(input [7:0] req, output [7:0] grant);\n" + "\n".join(equations) + "\nendmodule\n"

def arbiter_isolate():
    return """module arbiter(input [7:0] req, output [7:0] grant);
  assign grant = req & (~req + 8'd1);
endmodule
"""

def arbiter_groups():
    return """module arbiter(input [7:0] req, output [7:0] grant);
  wire [3:0] lo = req[3:0], hi = req[7:4];
  assign grant[3:0] = lo & (~lo + 4'd1);
  assign grant[7:4] = (hi & (~hi + 4'd1)) & {4{lo == 4'b0}};
endmodule
"""

def multiplier_operator():
    return """module multi_8bit(input [7:0] A,B, output [15:0] product);
  assign product = A * B;
endmodule
"""

def multiplier_split():
    return """module multi_8bit(input [7:0] A,B, output [15:0] product);
  wire [11:0] lo = A * B[3:0];
  wire [11:0] hi = A * B[7:4];
  assign product = {4'b0,lo} + {hi,4'b0};
endmodule
"""

def multiplier_tree():
    rows = [f"  wire [15:0] p{i} = ({{8'b0,A}} & {{16{{B[{i}]}}}}) << {i};" for i in range(8)]
    return """module multi_8bit(input [7:0] A,B, output [15:0] product);
""" + "\n".join(rows) + """
  assign product = ((p0+p1)+(p2+p3)) + ((p4+p5)+(p6+p7));
endmodule
"""

def shifter_operator():
    return """module barrel_shifter(input [7:0] in, input [2:0] ctrl, output [7:0] out);
  assign out = in >> ctrl;
endmodule
"""

def shifter_reorder():
    return """module barrel_shifter(input [7:0] in, input [2:0] ctrl, output [7:0] out);
  wire [7:0] s1=ctrl[0] ? {1'b0,in[7:1]} : in;
  wire [7:0] s2=ctrl[1] ? {2'b0,s1[7:2]} : s1;
  assign out=ctrl[2] ? {4'b0,s2[7:4]} : s2;
endmodule
"""

def shifter_decode():
    equations = [f"  assign out[{i}] = " + " | ".join(
        f"(in[{i+j}] & (ctrl == 3'd{j}))" for j in range(8-i)) + ";" for i in range(8)]
    return ("module barrel_shifter(input [7:0] in, input [2:0] ctrl, output [7:0] out);\n"
            + "\n".join(equations) + "\nendmodule\n")

def counter_bits():
    return """module counter_12(input rst_n,clk,valid_count, output reg [3:0] out);
  wire wrap = out == 4'd11;
  wire [3:0] toggle = {(&out[2:0]),(&out[1:0]),out[0],1'b1};
  always @(posedge clk or negedge rst_n)
    if (!rst_n) out <= 4'b0;
    else if (valid_count) out <= wrap ? 4'b0 : (out ^ toggle);
endmodule
"""

def counter_enable():
    return """module counter_12(input rst_n,clk,valid_count, output reg [3:0] out);
  wire wrap = valid_count && out == 4'd11;
  always @(posedge clk or negedge rst_n)
    if (!rst_n) out <= 4'b0;
    else out <= wrap ? 4'b0 : out + {3'b0,valid_count};
endmodule
"""

def fifo_flags():
    return """module fifo(input clk,rst,wr_en,rd_en, input [7:0] din,
    output reg [7:0] dout, output full,empty);
  reg [7:0] mem [0:3];
  reg [1:0] wptr,rptr;
  reg phase;
  wire push=wr_en & ~full, pop=rd_en & ~empty;
  assign full=(wptr==rptr) & phase;
  assign empty=(wptr==rptr) & ~phase;
  always @(posedge clk)
    if (rst) begin wptr<=0; rptr<=0; phase<=0; dout<=0; end
    else begin
      if (push) begin mem[wptr]<=din; wptr<=wptr+2'd1; end
      if (pop) begin dout<=mem[rptr]; rptr<=rptr+2'd1; end
      phase <= phase ^ (push & (&wptr)) ^ (pop & (&rptr));
    end
endmodule
"""

def fifo_shift_register():
    return """module fifo(input clk,rst,wr_en,rd_en, input [7:0] din,
    output reg [7:0] dout, output full,empty);
  reg [7:0] words [0:3];
  reg [2:0] count;
  wire push=wr_en & ~full, pop=rd_en & ~empty;
  wire [1:0] tail=count[1:0] - {1'b0,pop};
  assign full=(count==3'd4);
  assign empty=(count==3'd0);
  integer i;
  always @(posedge clk)
    if (rst) begin count<=0; dout<=0; end
    else begin
      if (pop) dout<=words[0];
      for(i=0;i<4;i=i+1)
        if (push && tail==i) words[i]<=din;
        else if (pop && i<3) words[i]<=words[(i+1)%4];
      if (push ^ pop) count<=pop ? count-3'd1 : count+3'd1;
    end
endmodule
"""

def replace_once(text, before, after):
    return text.replace(before, after, 1) if before in text else text

def alu_narrow_amount(text):
    if "wire [5:0] shift_imm" in text:
        return text
    lines = {
        "b << a;": "b << shift_imm;", "b >> a;": "b >> shift_imm;",
        "b_signed >>> a_signed;": "b_signed >>> shift_imm;",
    }
    if not all(key in text for key in lines):
        return text
    text = replace_once(text, "reg [32:0] res;",
                        "reg [32:0] res;\n    wire [5:0] shift_imm = (|a[31:6]) ? 6'd63 : a[5:0];")
    for before, after in lines.items():
        text = replace_once(text, before, after)
    return replace_once(text, "always @ (a or b or aluc)", "always @*")

def alu_share_compare(text):
    if "wire signed_lt" in text:
        return text
    text = replace_once(text, "reg [32:0] res;", "reg [32:0] res;\n"
                        "    wire signed_lt = a_signed < b_signed;\n    wire unsigned_lt = a < b;")
    text = replace_once(text, "(a_signed < b_signed) : (a < b)", "signed_lt : unsigned_lt")
    text = replace_once(text, "res <= a_signed < b_signed ? 1 : 0;", "res <= {32'b0,signed_lt};")
    text = replace_once(text, "res <= a < b ? 1 : 0;", "res <= {32'b0,unsigned_lt};")
    return replace_once(text, "always @ (a or b or aluc)", "always @*")

def alu_share_arithmetic(text):
    if "wire [32:0] sum_diff" in text:
        return text
    text = replace_once(text, "reg [32:0] res;", """reg [32:0] res;
    wire signed_mode = aluc == ADD || aluc == SUB;
    wire sub_mode = aluc == SUB || aluc == SUBU;
    wire [32:0] ext_a = {signed_mode & a[31], a};
    wire [32:0] ext_b = {signed_mode & b[31], b};
    wire [32:0] sum_diff = ext_a + (ext_b ^ {33{sub_mode}}) + {32'b0,sub_mode};""")
    for expr in ("a_signed + b_signed", "a + b", "a_signed - b_signed", "a - b"):
        text = replace_once(text, "res <= " + expr + ";", "res <= sum_diff;")
    return replace_once(text, "always @ (a or b or aluc)", "always @*")

def alu_merge_shift_modes(text):
    if "wire [5:0] shift_imm" not in text or "wire [5:0] shift_amount" in text:
        return text
    equations = {
        "b << shift_imm": "left_shift", "b << a[4:0]": "left_shift",
        "b >> shift_imm": "logical_shift", "b >> a[4:0]": "logical_shift",
        "b_signed >>> shift_imm": "arithmetic_shift", "b_signed >>> a_signed[4:0]": "arithmetic_shift",
    }
    text = replace_once(text, "always @*", """wire is_variable_shift = aluc == SLLV || aluc == SRLV || aluc == SRAV;
    wire [5:0] shift_amount = is_variable_shift ? {1'b0,a[4:0]} : shift_imm;
    wire [32:0] left_shift = b << shift_amount;
    wire [32:0] logical_shift = b >> shift_amount;
    wire signed [32:0] arithmetic_shift = b_signed >>> shift_amount;
    always @*""")
    for expr, shared in equations.items():
        text = replace_once(text, "res <= " + expr + ";", "res <= " + shared + ";")
    return text

VARIANTS = {
    "alu": {"narrow_shift_amount": alu_narrow_amount, "share_compare": alu_share_compare,
            "share_arithmetic": alu_share_arithmetic, "merge_shift_modes": alu_merge_shift_modes},
    "arbiter": {"prefix_mask": arbiter_prefix, "isolate_lowest": arbiter_isolate, "grouped_priority": arbiter_groups},
    "multiplier": {"multiply_operator": multiplier_operator, "split_product": multiplier_split,
                   "balanced_partial_products": multiplier_tree},
    "shifter": {"shift_operator": shifter_operator, "reordered_mux_stages": shifter_reorder,
                "decoded_shift": shifter_decode},
    "counter": {"toggle_bits": counter_bits, "arithmetic_enable": counter_enable},
    "fifo": {"pointer_wrap_phase": fifo_flags, "shift_queue": fifo_shift_register},
}
GENERIC = {
    "merge_mux_inputs": "opt -full; opt_share; opt_clean",
    "reduce_widths": "opt_expr; wreduce; opt_clean",
    "merge_and_reduce": "opt -full; opt_share; wreduce; opt_clean",
}

def available_actions(name):
    return sorted(VARIANTS[name]) + (list(GENERIC) if name != "alu" else [])

def transform(name, rtl, action, work):
    if action in VARIANTS[name]:
        fn = VARIANTS[name][action]
        return fn(rtl) if name == "alu" else fn()
    write_candidate(work, rtl)
    script = (f"read_verilog -sv candidate.v; hierarchy -check -top {TOP[name]}; proc; "
              f"flatten; {GENERIC[action]}; write_verilog -noattr output.v")
    checked(["yosys", "-Q", "-T", "-p", script], cwd=work, timeout=45, log=work / "rewrite.log")
    return (work / "output.v").read_text()

def reward_and_transition(row, current_adp):
    if row["status"] == "NO_CHANGE":
        return -0.04, False
    if row["status"] != "MEASURED":
        return -0.50, False
    if not row["eligible_under_limits"]:
        return -0.20, False
    delta = (current_adp-row["adp"]) / row["baseline_adp"]
    return float(max(-0.4, min(1.0, delta - 0.01))), True

def actionable_rewrites(name, rtl, actions, known_noops):
    """Materialize cheap rewrites and skip actions that cannot change this source."""
    proposals = {}
    key = digest(rtl)
    for action in actions:
        if (key, action) in known_noops:
            continue
        fn = VARIANTS[name].get(action)
        if fn is None:
            proposals[action] = None
        else:
            proposal = fn(rtl) if name == "alu" else fn()
            if proposal != rtl:
                proposals[action] = proposal
    return proposals

def model_action_ranker(model_key):
    import torch
    tokenizer, model = load_project_model(model_key)
    scores_by_state = {}
    revision = getattr(model.config, "_commit_hash", None)

    def rank(name, rtl, actions):
        key = (name, digest(rtl), tuple(actions))
        if key in scores_by_state:
            return scores_by_state[key]
        labels = [chr(ord("A")+index) for index in range(len(actions))]
        token_ids = [tokenizer.encode(label, add_special_tokens=False) for label in labels]
        if any(len(ids) != 1 for ids in token_ids) or len({ids[0] for ids in token_ids}) != len(labels):
            raise ValueError("The selected tokenizer must encode each action label as one token.")
        menu = "\\n".join(f"{label}. {action.replace('_', ' ')}" for label, action in zip(labels, actions))
        prompt = (
            f"Choose the single action most likely to reduce mapped area-delay product. "
            f"Preserve functionality, interface and cycle timing. Reply with one uppercase label.\\n"
            f"Benchmark: {name}\\n{SPEC[name]}\\nActions:\\n{menu}\\nRTL:\\n{rtl}\\n"
        )
        inputs = model_inputs(prompt, model_key, tokenizer, model)
        with torch.inference_mode():
            logits = model(**inputs, use_cache=False, logits_to_keep=1).logits[0, -1]
            selected = logits[[ids[0] for ids in token_ids]].float().cpu().tolist()
        if not all(math.isfinite(value) for value in selected):
            raise RuntimeError("The model returned non-finite action scores.")
        ranked = dict(zip(actions, selected))
        scores_by_state[key] = ranked
        return ranked

    return rank, revision

def cached_candidate_check(row, rtl, cache):
    key = digest(rtl)
    if key not in cache:
        candidate_check(row, rtl)
        if row["status"] == "MEASURED":
            cache[key] = dict(row)
        return
    prior = cache[key]
    dest = Path(row["artifact_dir"])
    write_candidate(dest, rtl)
    reused = ("status", "error", "syntax", "compile", "interface", "simulation", "checks",
              "formal", "synth_status", "timing_status", "cell_count", "cell_mix", "area_um2",
              "delay_ns", "slack_ns", "adp", "area_improvement_pct", "delay_improvement_pct",
              "adp_improvement_pct", "adp_improved", "eligible_under_limits")
    row.update({field: prior[field] for field in reused})
    row.update(candidate_path=str(dest / "candidate.v"), source_sha256=key,
               verification_reused_from=prior["artifact_dir"])
    # Exact cached matches share the original proof, simulator and timing logs.
    write_json(dest / "verification_reuse.json", {"source": prior["artifact_dir"], "sha256": key})

def rng_tuple(value):
    return tuple(map(rng_tuple, value)) if isinstance(value, list) else value

def lazy_model_ranker(model_key):
    ranker = None
    def rank(*args):
        nonlocal ranker
        if ranker is None:
            ranker, _ = model_action_ranker(model_key)
        return ranker(*args)
    return rank

def run_rl(name, label, episodes, horizon, seed, method="q_learning", resume=None, *,
           model_key=None, ranker=None, model_revision=None, verification_cache=None, known_noops=None):
    if method not in {"q_learning", "random_policy"} or min(episodes, horizon) < 1:
        raise ValueError("Use q_learning/random_policy and positive episodes/horizon.")
    if model_key is not None and ranker is None:
        ranker = lazy_model_ranker(model_key)
    actions = available_actions(name)
    policy_version = "tabular-v2"
    policy_id = digest(CONFIG_ID + name + repr(actions) + policy_version + str(model_key)
                       + str(model_revision) + str(horizon))
    values = {}
    verification_cache = {} if verification_cache is None else verification_cache
    known_noops = set() if known_noops is None else known_noops
    model_name = model_key or method
    model_id = MODELS[model_key] if model_key else policy_version
    rng = random.Random(seed)
    run = Experiment(name, method, label, model=model_name, model_id=model_id,
                     model_revision=model_revision, policy_version=policy_version, actions=actions,
                     seed=seed, episodes=episodes, horizon=horizon,
                     resume=str(resume) if resume else None, alpha=0.45, gamma=0.9,
                     epsilon_start=0.85, epsilon_end=0.20)
    def save_policy():
        if method == "q_learning":
            write_json(run.directory / "policy.json", {"policy_id": policy_id, "benchmark": name,
                       "config_id": CONFIG_ID, "seed": seed, "actions": actions, "q_values": values})

    completed = run.committed(resumable_rl=True)
    best = run.restore_best(completed)
    attempt = len(completed)
    checkpoint = completed[-1]["_resume"] if completed else {}
    if checkpoint:
        values = checkpoint["q_values"]
        rng.setstate(rng_tuple(checkpoint["rng"]))
        known_noops.update(map(tuple, checkpoint["known_noops"]))
        for row in completed:
            if row["status"] == "MEASURED" and not row.get("verification_reused_from"):
                verification_cache.setdefault(row["source_sha256"], row)
        save_policy()
    elif resume:
        saved = read_json(resume)
        if method != "q_learning" or saved["policy_id"] != policy_id:
            raise ValueError("Resume with the same benchmark, setup, model, action set and horizon.")
        values = saved["q_values"]
    if run.is_complete:
        print(f"{name} {model_name} {label}: {attempt} saved attempts already complete.")
        return run
    if attempt:
        print(f"{name} {model_name} {label}: resuming after {attempt} saved attempts.")
    first_episode = checkpoint.get("episode", 0)
    first_step = checkpoint.get("step", 0)
    for episode in range(first_episode, episodes):
        offset = first_step if episode == first_episode else 0
        current_path = checkpoint.get("current_candidate") if offset else None
        rtl = Path(current_path).read_text() if current_path else REFERENCES[name]
        current_adp = checkpoint["current_adp"] if offset else BASELINES[name]["adp"]
        epsilon = 0.85 - 0.65 * episode / max(1, episodes-1)
        for step in range(offset, horizon):
            proposals = actionable_rewrites(name, rtl, actions, known_noops)
            if not proposals:
                break
            attempt += 1
            state = f"{horizon-step}:{digest(rtl)}"
            scores = values.setdefault(state, {})
            explore = method == "random_policy" or rng.random() < epsilon
            pool = list(proposals)
            if not explore:
                top_q = max(scores.get(action, 0.0) for action in pool)
                pool = [action for action in pool if scores.get(action, 0.0) == top_q]
            dest = run.attempt_dir(attempt)
            run.retry_slot(dest)
            row = init_row(name, method, run, attempt, dest, model_name, model_id, seed)
            prior = {}
            model_seconds = 0.0
            try:
                if ranker is not None and method == "q_learning" and len(pool) > 1 and (
                        not explore or rng.random() < 0.75):
                    ranking_started = time.monotonic()
                    prior = ranker(name, rtl, list(proposals))
                    model_seconds = time.monotonic() - ranking_started
                    action = max(pool, key=prior.__getitem__)
                else:
                    action = rng.choice(pool)
            except Exception:
                trace = traceback.format_exc()
                (dest / "error.log").write_text(trace)
                row.update(status="MODEL_PRIOR_ERROR", error=trace[-4500:], policy_state=state)
                finish_row(row)
                raise
            row.update(action=action, policy_state=state)
            detail = {"episode": episode+1, "step": step+1, "epsilon": epsilon,
                      "action": action, "start_adp": current_adp, "state": state,
                      "model_action_scores": prior, "model_prior_seconds": round(model_seconds, 4)}
            proposal = rtl
            try:
                start = time.monotonic()
                proposal = proposals[action]
                if proposal is None:
                    proposal = stage(dest / "transform", lambda work: transform(name, rtl, action, work))
                row["generation_seconds"] = round(model_seconds + time.monotonic() - start, 4)
                if proposal == rtl:
                    known_noops.add((digest(rtl), action))
                    row.update(status="NO_CHANGE", error="Action produced identical RTL.")
                else:
                    cached_candidate_check(row, proposal, verification_cache)
                    best = record_best(run, row, best)
            except Exception:
                trace = traceback.format_exc()
                (dest / "error.log").write_text(trace)
                row.update(status="EVALUATION_ERROR", error=trace[-4500:])
            reward, move = reward_and_transition(row, current_adp)
            if move:
                rtl, current_adp = proposal, row["adp"]
                current_path = row["candidate_path"]
            if method == "q_learning":
                next_state = f"{horizon-step-1}:{digest(rtl)}"
                future = values.get(next_state, {})
                remaining = actionable_rewrites(name, rtl, actions, known_noops)
                continuation = max((future.get(a, 0.0) for a in remaining), default=0.0) if step+1 < horizon else 0.0
                old = scores.get(action, 0.0)
                scores[action] = old + 0.45 * (reward + 0.9 * continuation - old)
            row["reward"] = reward
            if row["best_allowed_adp"] is None:
                row["best_allowed_adp"] = best["adp"] if best else row["baseline_adp"]
            write_json(dest / "transition.json", {**detail, "move": move, "reward": reward,
                                                  "result_adp": current_adp, "next_rtl_sha256": digest(rtl)})
            next_step = (step+1) % horizon
            row["_resume"] = {
                "episode": episode + int(next_step == 0), "step": next_step,
                "current_candidate": current_path, "current_adp": current_adp,
                "q_values": values, "rng": rng.getstate(), "known_noops": sorted(known_noops),
            }
            finish_row(row)
            if row["new_best"]:
                run.save_best(row)
            save_policy()
            print(f"{name} {model_name} ep={episode+1} step={step+1} {action}: "
                  f"{row['status']} reward={reward:.4f} ADP={row['adp']}", flush=True)
    run.complete()
    return run

def run_rl_batch(modules, episodes, horizon, seeds, random_control=False, *, model_key=None, workers=1):
    from concurrent.futures import ThreadPoolExecutor

    usable = [name for name in modules if name in READY_BENCHMARKS]
    if set(usable) != set(modules):
        print("Blocked reference(s):", sorted(set(modules)-set(usable)))
    if not usable:
        raise ValueError("Select a benchmark that passed reference preflight.")
    if workers < 1 or (model_key and workers != 1):
        raise ValueError("Use workers >= 1. Model-guided RL requires workers=1.")
    ranker = lazy_model_ranker(model_key) if model_key else None
    def run_benchmark(name):
        verification_cache = {}
        for seed in seeds:
            run_rl(name, seed, episodes, horizon, seed, model_key=model_key, ranker=ranker,
                   verification_cache=verification_cache,
                   known_noops=set())
            if random_control:
                run_rl(name, seed, episodes, horizon, seed, method="random_policy",
                       verification_cache=verification_cache, known_noops=set())
    try:
        if workers == 1:
            for name in usable:
                run_benchmark(name)
        else:
            with ThreadPoolExecutor(max_workers=workers) as pool:
                list(pool.map(run_benchmark, usable))
    finally:
        export_results()
    return all_attempts()


In [ ]:
MODULES = READY_BENCHMARKS
EPISODES = 5
HORIZON = 5
SEEDS = (2,)
RANDOM_CONTROL = False
attempts_table = run_rl_batch(MODULES, EPISODES, HORIZON, SEEDS, RANDOM_CONTROL, model_key=MODEL_KEY)
display(attempts_table.tail(12)[["benchmark", "method", "action", "status", "simulation",
                                "formal", "adp", "reward", "adp_improvement_pct"]])


In [ ]:
def pass_rate_report(table):
    groups = ["workflow", "model", "model_id", "method", "benchmark"]
    rows = []
    for keys, batch in table.groupby(groups, dropna=False):
        candidates = batch.loc[batch["source_sha256"].notna() & batch["status"].ne("NO_CHANGE")]
        unique = (candidates.assign(
            simulated=candidates["simulation"].eq("PASS"),
            proved=candidates["assurance"].eq("FORMALLY_VERIFIED"),
        ).groupby("source_sha256")[["simulated", "proved"]].any())
        passes = int(candidates["simulation"].eq("PASS").sum())
        proof_passes = int(candidates["assurance"].eq("FORMALLY_VERIFIED").sum())
        rows.append(dict(zip(groups, keys), action_attempts=len(batch),
                         no_changes=int(batch["status"].eq("NO_CHANGE").sum()),
                         candidates=len(candidates), simulation_passes=passes, formally_verified=proof_passes,
                         candidate_sim_pass_pct=100*passes/len(candidates) if len(candidates) else None,
                         attempt_sim_pass_pct=100*passes/len(batch),
                         unique_candidates=len(unique), unique_simulation_passes=int(unique["simulated"].sum()),
                         unique_formally_verified=int(unique["proved"].sum()),
                         verification_cache_hits=int(batch["verification_reused_from"].notna().sum())))
    return pd.DataFrame(rows, columns=groups + [
        "action_attempts", "no_changes", "candidates", "simulation_passes", "formally_verified",
        "candidate_sim_pass_pct", "attempt_sim_pass_pct", "unique_candidates",
        "unique_simulation_passes", "unique_formally_verified", "verification_cache_hits"])

def comparison_report(compare_other_workflow=False, make_plot=True):
    table = export_results()
    if compare_other_workflow:
        other = OTHER_PROJECT / "results" / "verified_rtl_v4" / OTHER_WORKFLOW
        if other.is_dir():
            table = pd.concat([table, all_attempts(other)], ignore_index=True)
    table = table.loc[table["config_id"].eq(CONFIG_ID)]
    by_benchmark = summarize(table)
    save_csv(OUTPUT / "comparison_by_benchmark.csv", by_benchmark)
    save_csv(OUTPUT / "comparison_by_run.csv", summarize(table, by_run=True))
    stage_counts = (table.groupby(["workflow", "model", "method", "benchmark", "status"], dropna=False)
                    .size().rename("attempts").reset_index())
    save_csv(OUTPUT / "outcome_counts.csv", stage_counts)
    save_csv(OUTPUT / "pass_rates.csv", pass_rate_report(table))
    candidates = table.loc[table["competition_eligible"].eq(True)]
    grouped = ["workflow", "model", "method", "benchmark"]
    winners = candidates.sort_values(["adp", "attempt_id"]).drop_duplicates(grouped)
    if make_plot and not winners.empty:
        import matplotlib.pyplot as plt
        plotting = winners.assign(series=winners["model"]+" / "+winners["method"])
        pivot = plotting.pivot(index="benchmark", columns="series", values="adp_improvement_pct")
        axes = pivot.plot.bar(figsize=(max(8, len(pivot)*1.8), 4.8))
        axes.set_ylabel("Best formally verified ADP reduction (%)")
        axes.set_xlabel("")
        axes.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
        axes.figure.tight_layout()
        axes.figure.savefig(OUTPUT / "verified_adp_comparison.png", dpi=170, bbox_inches="tight")
        plt.show()
    if table.empty:
        print("No attempts under this measurement configuration yet.")
    return by_benchmark

def recent_attempts(benchmark=None, limit=10, with_logs=False):
    table = all_attempts()
    if benchmark:
        table = table.loc[table["benchmark"].eq(benchmark)]
    result = table.sort_values("utc").tail(limit)
    if with_logs:
        for row in result.itertuples():
            print(row.benchmark, row.status, row.artifact_dir)
            if row.error:
                print(row.error)
    return result[["model", "benchmark", "method", "run", "attempt", "status",
                   "simulation", "formal", "audit_status", "area_um2", "delay_ns",
                   "adp", "adp_improvement_pct", "artifact_dir"]]


In [ ]:
display(comparison_report(compare_other_workflow=True))
display(recent_attempts())
